# Module 1 — Data Ingestion

Exploratory notebook for `analysis/fetch.py`. The goal of this module is to pull
and cache historical **adjusted close** prices for a configurable list of ETFs.

**Key concepts:** the `yfinance` API, `pandas.DataFrame`, datetime indexing,
file I/O with pandas, and handling missing data.

> Convention (see `SPEC.md`): explore here first, then refactor the working
> logic into `analysis/fetch.py`. This notebook is kept as an honest artifact of
> the process.

## Setup

In [ ]:
import sys
from pathlib import Path

# Make the project root importable when running from notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import yfinance as yf

from analysis import fetch
from analysis.config import DEFAULT_TICKERS, DEFAULT_START

DEFAULT_TICKERS, DEFAULT_START

## 1. What does raw `yfinance` output look like?

We download with `auto_adjust=True` so the `Close` column is already the
*adjusted* close (dividends and splits folded in) — exactly what we want for
return calculations. For multiple tickers, the columns come back as a
`(field, ticker)` MultiIndex.

In [ ]:
raw = yf.download(
    DEFAULT_TICKERS,
    start=DEFAULT_START,
    end="2023-01-01",
    auto_adjust=True,
    progress=False,
    group_by="column",
)
raw.head()

In [ ]:
# The columns are a MultiIndex: (field, ticker)
raw.columns[:6].tolist()

## 2. Reduce to one adjusted-close column per ticker

This is exactly what `fetch.fetch_prices` does under the hood.

In [ ]:
prices = fetch.fetch_prices(DEFAULT_TICKERS, start=DEFAULT_START, end="2023-01-01")
prices.head()

## 3. What do `.info()`, `.describe()`, `.head()` tell us?

In [ ]:
prices.info()

In [ ]:
prices.describe()

## 4. Date boundaries — weekends & holidays

Markets are closed on weekends and holidays, so the index only contains trading
days. Notice the gaps between Friday and Monday below.

In [ ]:
prices.index[:7].tolist()

In [ ]:
# Count trading days per year — roughly ~252, the standard assumption.
prices.groupby(prices.index.year).size()

## 5. Handling a ticker with less history

If one ETF is newer than the others, the early rows are `NaN` for that column.
`fetch_prices` drops rows that are NaN for *every* ticker, but keeps rows where
only some are missing — downstream modules decide how to align. Here we look at
a young ETF (`AVUV`, inception 2019) alongside `VTI`.

In [ ]:
mixed = fetch.fetch_prices(["VTI", "AVUV"], start=DEFAULT_START, end="2023-01-01")
print("First non-NaN AVUV date:", mixed["AVUV"].first_valid_index())
mixed.loc[mixed["AVUV"].isna()].head()

## 6. Quick visual sanity check

Normalize each series to its first value to compare growth on one axis.

In [ ]:
import matplotlib.pyplot as plt

normalized = prices / prices.iloc[0]
normalized.plot(figsize=(10, 5), title="Growth of $1 (adjusted close)")
plt.ylabel("Growth multiple")
plt.tight_layout()

## 7. Caching with `load_or_fetch`

`load_or_fetch` writes one CSV per ticker to `data/raw/` and reuses them until
the requested `end` date is newer than the cache. The first call hits the
network; the second reads from disk.

In [ ]:
cached = fetch.load_or_fetch(DEFAULT_TICKERS, start=DEFAULT_START, end="2023-01-01")
cached.tail()

## Checkpoint

**What is a pandas DataFrame and how is it different from a 2D NumPy array?**

A `DataFrame` is a 2D table with *labeled* axes: a `DatetimeIndex` of trading
days and named columns (one per ticker). Unlike a raw 2D NumPy array, it carries
those labels, allows mixed dtypes per column, aligns operations on the index
automatically, and handles missing data (`NaN`) natively — all of which we lean
on throughout this project.